# Проверка train_relabel_v2_synth_v1

Сверяем объединённый train с версиями микроправок и синтеза. В таблицах видны состав, причины исключения строк и число примеров для каждой метки.

In [1]:
import json
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import pyarrow.parquet as pq

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'data/processed/train_relabel_v2_synth_v1').is_dir())
sys.path.insert(0, str(root / 'dataset_review'))
from build_train_relabel_v2_synth_v1 import duplicate_key, gold_overlap_key, key, label_counts, labels_for, sha256

version = root / 'data/processed/train_relabel_v2_synth_v1'
manifest = json.loads((version / 'manifest.json').read_text())
for info in [manifest['dataset'], manifest['excluded_real'], *manifest['sources'].values()]:
    assert sha256(root / info['path']) == info['sha256'], info['path']

rows = pq.read_table(root / manifest['dataset']['path']).to_pylist()
by_origin = {origin: [row for row in rows if row['origin'] == origin]
             for origin in ('real', 'synthetic')}
assert len(rows) == manifest['dataset']['records']
assert dict(Counter(row['origin'] for row in rows)) == manifest['records_by_origin']
assert len({row['item_id'] for row in rows}) == len(rows)
description_keys = {duplicate_key(row['description']) for row in rows}
assert len(description_keys) == len(rows) and '' not in description_keys

display(pd.Series(manifest['records_by_origin'], name='Строк'))
display(pd.DataFrame({origin: label_counts(group) for origin, group in by_origin.items()}
                     | {'total': label_counts(rows)}).rename_axis('Метка'))
print('SHA-256 совпадают, дубликатов и пустых описаний нет.')

real         4214
synthetic    1097
Name: Строк, dtype: int64

,real,synthetic,total
Метка,,,
102,1407,277,1684
103,1891,308,2199
104,1825,465,2290
105,1814,358,2172
106,2139,86,2225


SHA-256 совпадают, дубликатов и пустых описаний нет.


In [2]:
real_source = {row['item_id']: row for row in
               pq.read_table(root / manifest['sources']['real']['path']).to_pylist()}
synth_source = {row['synth_task_id']: row for row in
                pq.read_table(root / manifest['sources']['synthetic']['path']).to_pylist()
                if row['origin'] == 'synthetic'}

assert all({name: row[name] for name in real_source[row['item_id']]} == real_source[row['item_id']]
           for row in by_origin['real'])
assert {row['synth_task_id'] for row in by_origin['synthetic']} == synth_source.keys()
for row in by_origin['synthetic']:
    source = synth_source[row['synth_task_id']]
    for field in ('item_id', 'title', 'description', 'found_microcategories', 'synth_checked'):
        assert row[field] == source[field], (row['synth_task_id'], field)
    assert row['synth_checked'] is True
    assert row['source_item_id'] == -source['source_line']
    assert row['source_line'] == real_source[row['source_item_id']]['source_line']

excluded = pd.read_csv(version / 'excluded_real.csv')
assert len(excluded) == excluded.item_id.nunique() == manifest['excluded_real']['records']
assert set(excluded.item_id) == real_source.keys() - {row['item_id'] for row in by_origin['real']}
all_five = sum(len(labels_for(row)) == 5 for row in by_origin['real'])
assert all_five == manifest['transformations']['all_five']['after_cap']
display(excluded.groupby('reason').size().rename('Исключено строк'))
print(f'Текст и метки сохранены во всех взятых строках. Реальных строк со всеми пятью метками: {all_five}.')

reason
all_five_cap    523
test_overlap     12
Name: Исключено строк, dtype: int64

Текст и метки сохранены во всех взятых строках. Реальных строк со всеми пятью метками: 800.


In [3]:
normalize = gold_overlap_key()
train_exact = {key(row['description']) for row in rows}
train_normalized = {normalize(row['description']) for row in rows} - {''}
overlaps = {}
for name in ('test_gold', 'public_test'):
    test = pq.read_table(root / manifest['sources'][name]['path'], columns=['description'])
    overlaps[name] = sum(key(text) in train_exact or normalize(text) in train_normalized
                         for text in test.column('description').to_pylist())

assert not any(overlaps.values())
display(pd.Series(overlaps, name='Строки теста, совпавшие с train'))

test_gold      0
public_test    0
Name: Строки теста, совпавшие с train, dtype: int64